# 方法2最小实验：Block/Site 级 Memorization Patching（双模型对照）

本 notebook 目标：用仓库现有接口，**同时加载 standard 与 attnres 两个训练 checkpoint**，
并做 block/site 级 clean vs corrupted vs patched/ablated 对照。

范围刻意最小：
- 只做方法2（Model Attribution）
- 不做 head-level / neuron-level
- 不重写训练框架


In [ ]:
import json
import sys
from copy import deepcopy
from pathlib import Path

import torch
import matplotlib.pyplot as plt

# 定位仓库根目录（兼容从仓库根或 notebook 子目录启动）
CANDIDATES = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = None
for p in CANDIDATES:
    if (p / "toygpt2").exists():
        REPO_ROOT = p
        break
if REPO_ROOT is None:
    raise RuntimeError("未找到仓库根目录（应包含 toygpt2 目录）")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT:", REPO_ROOT)
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)


## Checkpoint 选择策略（双模型）

本 notebook **同时加载两份训练完成 checkpoint**：
- standard: `/home/cody/LLM_ML_CODE_DATA/FINAL_report_project/toygpt2_runs/tinystories_dual/standard/ckpt_standard_last.pt`
- attnres: `/home/cody/LLM_ML_CODE_DATA/FINAL_report_project/toygpt2_runs/tinystories_dual/attnres/ckpt_attnres_last.pt`

后续所有输出都用 `[standard]` / `[attnres]` 标注。


In [ ]:
from scripts.evaluate import load_checkpoint

CHECKPOINTS = {
    "standard": str(REPO_ROOT / "toygpt2_runs" / "tinystories_dual" / "standard" / "ckpt_standard_last.pt"),
    "attnres": str(REPO_ROOT / "toygpt2_runs" / "tinystories_dual" / "attnres" / "ckpt_attnres_last.pt"),
}
MODEL_ORDER = ["standard", "attnres"]
device = torch.device("cpu")

models = {}
experiments = {}
checkpoints = {}
load_errors = {}

for model_name in MODEL_ORDER:
    ckpt_path = Path(CHECKPOINTS[model_name])
    try:
        model, experiment, checkpoint = load_checkpoint(ckpt_path, device=device)
        model = model.to(device).eval()
        models[model_name] = model
        experiments[model_name] = experiment
        checkpoints[model_name] = checkpoint

        ckpt_model_type = str(checkpoint.get("model_type"))
        print(f"[{model_name}] checkpoint: {ckpt_path}")
        print(f"[{model_name}] model_type(in_ckpt): {ckpt_model_type}")
        print(f"[{model_name}] step: {checkpoint.get('step')}")
        print(f"[{model_name}] n_layer={experiment.model.n_layer} n_head={experiment.model.n_head} n_embd={experiment.model.n_embd}")
        print(f"[{model_name}] vocab_size={experiment.model.vocab_size} block_size={experiment.model.block_size}")
        print(f"[{model_name}] device={next(model.parameters()).device}")
        if ckpt_model_type != model_name:
            print(f"[{model_name}] WARNING: checkpoint model_type 与期望不一致")
        print("-" * 80)
    except Exception as e:
        load_errors[model_name] = repr(e)
        print(f"[{model_name}] LOAD FAILED: {e!r}")

if load_errors:
    raise RuntimeError(f"以下模型加载失败: {load_errors}")


## 准备小样本输入（clean vs corrupted）

为保证对照公平，两个模型使用**同一份 token id 输入**。
输入按两模型共有约束构造：
- `shared_vocab_size = min(vocab_size_standard, vocab_size_attnres)`
- `shared_block_size = min(block_size_standard, block_size_attnres)`

不加载外部 GPT-2 或 tokenizer。


In [ ]:
from data.data import build_dataloaders
from interp.memorization_runner import MemorizationPatchingRunner

runners = {name: MemorizationPatchingRunner(models[name]) for name in MODEL_ORDER}

shared_block_size = min(experiments[name].model.block_size for name in MODEL_ORDER)
shared_vocab_size = min(experiments[name].model.vocab_size for name in MODEL_ORDER)
seq_len = int(min(8, shared_block_size))

if seq_len < 4:
    raise ValueError(f"shared_block_size={shared_block_size} 过小，至少需要 >=4")
if shared_vocab_size < 2:
    raise ValueError(f"shared_vocab_size={shared_vocab_size} 过小，至少需要 >=2")

sample_model_config = deepcopy(experiments[MODEL_ORDER[0]].model)
sample_model_config.block_size = shared_block_size
tiny_cfg = deepcopy(experiments[MODEL_ORDER[0]].data)
tiny_cfg.dataset_type = "tinystories"
tiny_cfg.block_stride = max(1, sample_model_config.block_size)
if tiny_cfg.train_texts is None:
    tiny_cfg.train_texts = 1024
if tiny_cfg.val_texts is None:
    tiny_cfg.val_texts = 256

train_loader, _ = build_dataloaders(
    model_config=sample_model_config,
    data_config=tiny_cfg,
    batch_size=1,
    num_workers=0,
    seed=0,
    distributed=False,
    verbose=True,
)
inputs, targets = next(iter(train_loader))
full_sample = torch.cat([inputs, targets[:, -1:].clone()], dim=1).to(device=device, dtype=torch.long)
clean_tokens = full_sample[:, :seq_len].clone()
if int(clean_tokens.max().item()) >= shared_vocab_size:
    clean_tokens = clean_tokens % shared_vocab_size
corrupted_tokens = clean_tokens.clone()
corrupt_position = min(2, seq_len - 2)
corrupted_tokens[0, corrupt_position] = (corrupted_tokens[0, corrupt_position] + 1) % shared_vocab_size

sample_info = {
    "source": "tinystories_train_loader",
    "seq_len": int(seq_len),
    "shared_vocab_size": int(shared_vocab_size),
    "shared_block_size": int(shared_block_size),
    "corrupt_position": int(corrupt_position),
    "vocab_by_model": {name: int(experiments[name].model.vocab_size) for name in MODEL_ORDER},
    "block_by_model": {name: int(experiments[name].model.block_size) for name in MODEL_ORDER},
}

print("sample_info:", json.dumps(sample_info, ensure_ascii=False, indent=2))
print("clean_tokens:", clean_tokens.tolist())
print("corrupted_tokens:", corrupted_tokens.tolist())

target_position = clean_tokens.size(1) - 1
target_token_id = int(clean_tokens[0, target_position].item())
print("target_position:", target_position, "target_token_id:", target_token_id)


## Clean forward + cache 可观测位点（双模型）

分别打印两个模型的 cache keys，并计算可 patch 位点交集（common patch sites）。


In [ ]:
from interp.analysis_adapter import AnalysisAdapter

clean_outputs_by_model = {}
cache_by_model = {}
trace_by_model = {}
candidate_sites_by_model = {}

with torch.no_grad():
    for model_name in MODEL_ORDER:
        outputs = models[model_name](
            clean_tokens,
            return_intermediates=True,
            return_attn=True,
            return_cache=True,
        )
        clean_outputs_by_model[model_name] = outputs

        cache_obj = outputs["cache"]
        cache_dict = cache_obj.data
        cache_by_model[model_name] = cache_dict
        cache_keys = sorted(cache_dict.keys())

        candidate_sites = [
            key for key in cache_keys if key.endswith(("attn_out", "mlp_out", "output"))
        ]
        candidate_sites_by_model[model_name] = candidate_sites

        trace = AnalysisAdapter.from_model_outputs(outputs)
        trace_by_model[model_name] = trace

        print(f"[{model_name}] forward output keys: {sorted(outputs.keys())}")
        print(f"[{model_name}] cache key count: {len(cache_keys)}")
        print(f"[{model_name}] 前 20 个 cache keys:")
        for key in cache_keys[:20]:
            print("  ", key)
        print(f"[{model_name}] candidate patch sites (前 20):")
        for key in candidate_sites[:20]:
            print("  ", key)
        print(f"[{model_name}] trace layers: {list(trace.layers())}")
        print(f"[{model_name}] layer0 block_type: {trace.layer(0).block_type}")
        print("-" * 80)

common_patch_sites = None
for model_name in MODEL_ORDER:
    sites = set(candidate_sites_by_model[model_name])
    common_patch_sites = sites if common_patch_sites is None else (common_patch_sites & sites)
common_patch_sites = sorted(common_patch_sites) if common_patch_sites is not None else []

print("common_patch_sites:")
for key in common_patch_sites[:40]:
    print(" ", key)

if not common_patch_sites:
    raise RuntimeError("两个模型没有可用的共同 patch site。")


## 选择 patch site（双模型可比）

优先使用 `blocks.0.attn_out`；若不在交集中则退化为第一个 common site。
这样 standard 与 attnres 都在同一 site 上做对照。


In [ ]:
from interp.ablation import zero_ablation_override

preferred_patch_site = "blocks.0.attn_out"
patch_site = preferred_patch_site if preferred_patch_site in common_patch_sites else common_patch_sites[0]
print("selected patch_site:", patch_site)

def token_logprob(outputs, pos, token_id):
    logits = outputs["logits"]
    return float(torch.log_softmax(logits, dim=-1)[0, pos, token_id].item())

results_by_model = {}
corrupted_outputs_by_model = {}
ablated_outputs_by_model = {}
summary_by_model = {}

for model_name in MODEL_ORDER:
    cache = cache_by_model[model_name]
    if patch_site not in cache:
        raise KeyError(f"[{model_name}] patch_site 不在 cache 中: {patch_site}")

    runner = runners[model_name]
    result_patch = runner.run(
        clean_input=clean_tokens,
        corrupted_input=corrupted_tokens,
        patch_site=patch_site,
        target_position=target_position,
        target_token_id=target_token_id,
        metric="logprob",
    )

    with torch.no_grad():
        corrupted_outputs = models[model_name](
            corrupted_tokens,
            return_intermediates=True,
            return_cache=True,
        )
        ablated_outputs = models[model_name](
            corrupted_tokens,
            return_intermediates=True,
            return_cache=True,
            activation_overrides=zero_ablation_override(patch_site),
        )

    clean_lp = float(result_patch.baseline_clean_score.item())
    corr_lp = float(result_patch.baseline_corrupted_score.item())
    patch_lp = float(result_patch.patched_score.item())
    ablate_lp = token_logprob(ablated_outputs, target_position, target_token_id)

    results_by_model[model_name] = result_patch
    corrupted_outputs_by_model[model_name] = corrupted_outputs
    ablated_outputs_by_model[model_name] = ablated_outputs
    summary_by_model[model_name] = {
        "clean": clean_lp,
        "corrupted": corr_lp,
        "patched": patch_lp,
        "ablated": ablate_lp,
        "effect": float(result_patch.effect_size.item()),
    }

    print(f"[{model_name}] patch_site: {patch_site}")
    print(f"[{model_name}] clean logprob: {clean_lp:.6f}")
    print(f"[{model_name}] corrupted logprob: {corr_lp:.6f}")
    print(f"[{model_name}] patched logprob: {patch_lp:.6f}")
    print(f"[{model_name}] zero-ablated logprob: {ablate_lp:.6f}")
    print(f"[{model_name}] patch effect (patched - corrupted): {float(result_patch.effect_size.item()):+.6f}")
    print("-" * 80)


## 最小指标：目标 token 分数与 top-k 变化（按模型分组）

每个模型分别输出 clean/corrupted/patched/ablated 的目标 logprob 与 top-k。


In [ ]:
def topk_tokens(outputs, pos, k=5):
    logits = outputs["logits"][0, pos]
    probs = torch.softmax(logits, dim=-1)
    vals, idx = torch.topk(probs, k=k)
    return [(int(i), float(v)) for i, v in zip(idx, vals)]

rows = []

for model_name in MODEL_ORDER:
    with torch.no_grad():
        clean_outputs_eval = models[model_name](clean_tokens)
        corr_outputs_eval = models[model_name](corrupted_tokens)
        patched_outputs_eval = models[model_name](
            corrupted_tokens,
            activation_overrides={patch_site: clean_outputs_by_model[model_name]["cache"][patch_site].clone()},
        )

    r = results_by_model[model_name]
    clean_lp = float(r.baseline_clean_score.item())
    corr_lp = float(r.baseline_corrupted_score.item())
    patch_lp = float(r.patched_score.item())
    ablate_lp = float(summary_by_model[model_name]["ablated"])

    rows.extend([
        {
            "model": model_name,
            "condition": "clean",
            "target_logprob": clean_lp,
            "top5": topk_tokens(clean_outputs_eval, target_position),
        },
        {
            "model": model_name,
            "condition": "corrupted",
            "target_logprob": corr_lp,
            "top5": topk_tokens(corr_outputs_eval, target_position),
        },
        {
            "model": model_name,
            "condition": "patched_from_clean",
            "target_logprob": patch_lp,
            "top5": topk_tokens(patched_outputs_eval, target_position),
        },
        {
            "model": model_name,
            "condition": "zero_ablated",
            "target_logprob": ablate_lp,
            "top5": topk_tokens(ablated_outputs_by_model[model_name], target_position),
        },
    ])

print(json.dumps(rows, indent=2, ensure_ascii=False))


## 可视化（双模型对照）

左图：两模型在四种条件下的目标 token logprob。
右图：两模型在同一批 patch sites 上的 effect 对比。


In [ ]:
sweep_sites = common_patch_sites[:6]
if patch_site not in sweep_sites:
    sweep_sites = [patch_site] + [s for s in sweep_sites if s != patch_site]

if not sweep_sites:
    raise RuntimeError("没有可用于 sweep 的 patch site。")

site_effects_by_model = {}
for model_name in MODEL_ORDER:
    sweep = runners[model_name].run_site_sweep(
        clean_input=clean_tokens,
        corrupted_input=corrupted_tokens,
        patch_sites=sweep_sites,
        target_position=target_position,
        target_token_id=target_token_id,
        metric="logprob",
    )
    site_effects_by_model[model_name] = [float(v.item()) for v in sweep.effect_by_site[:, 0]]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 左图：双模型四条件对照
conditions = ["clean", "corrupted", "patched", "ablated"]
x_labels = []
values = []
for model_name in MODEL_ORDER:
    for cond in conditions:
        x_labels.append(f"{model_name}\n{cond}")
        values.append(float(summary_by_model[model_name][cond]))

axes[0].bar(range(len(values)), values)
axes[0].set_xticks(range(len(values)))
axes[0].set_xticklabels(x_labels, rotation=20)
axes[0].set_title(f"Target logprob @pos={target_position}, token={target_token_id}")

# 右图：双模型 site effect 对比
base_x = list(range(len(sweep_sites)))
n_models = len(MODEL_ORDER)
bar_width = 0.8 / max(1, n_models)
for mi, model_name in enumerate(MODEL_ORDER):
    offsets = [x - 0.4 + bar_width / 2 + mi * bar_width for x in base_x]
    axes[1].bar(offsets, site_effects_by_model[model_name], width=bar_width, label=model_name)

axes[1].set_xticks(base_x)
axes[1].set_xticklabels(sweep_sites, rotation=30, ha="right")
axes[1].axhline(0.0, color="black", linewidth=1)
axes[1].set_title("Patch effect by site (dual-model)")
axes[1].legend()

plt.tight_layout()
plt.show()

print("site_effects_by_model:")
for model_name in MODEL_ORDER:
    print(f"[{model_name}]")
    for site, effect in zip(sweep_sites, site_effects_by_model[model_name]):
        print(f"  {site:36s} {effect:+.6f}")


## 结果解释（最小结论）

这份 notebook 实际完成了：
1. 同时加载 standard 与 attnres 两个训练 checkpoint
2. 用同一 clean/corrupted 输入做双模型对照
3. 在共同 patch site 上分别执行 patching 与 zero ablation
4. 输出按模型分组的分数表与可视化

它属于方法2（Model Attribution）的原因：
- 干预发生在 transformer block/site 层级
- 明确比较干预前后目标 token 分数变化
- 支持按层/位点 sweep

当前结果能说明：
- 在相同输入与相同 site 条件下，standard 与 attnres 的 patch effect 可直接对照。

当前结果还不能说明：
- 论文级普适因果结论（样本量、任务协议、统计检验仍不足）。
